# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
# Finding 1: The Freshness Multiplier (31-90 day window is the strongest stable freshness band)
# Methodology Question: Where does the "freshness" label come from? Is it based on any update, or only substantial content rewrites? Does the validation design control for the fact that high-performing pages are more likely to receive routine maintenance (survivorship bias)?

# Finding 2: Engagement and Visibility Move Together (High scroll + high engagement = +11.2 health points)
# Methodology Question: Does the validation design support causation or just correlation? Are pages with low scroll and low engagement less visible because they don't match the query well, meaning relevance drives both, rather than engagement driving visibility?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Features
num_features = ['search_volume', 'cpc', 'word_count', 'char_count', 'impressions_90d',
                'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
                'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
                'days_with_impressions', 'days_with_sessions', 'content_age_days',
                'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 
                'scroll_rate', 'ai_traffic_pct']
cat_features = ['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used']

X = df[num_features + cat_features]
y = df['is_declining_label']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), 
                          ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
    ])

rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, max_depth=10, n_estimators=100))
])

# 1. Random Split (Dishonest)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)
rf.fit(X_train_r, y_train_r)
auc_random = roc_auc_score(y_test_r, rf.predict_proba(X_test_r)[:, 1])

# 2. Grouped Split (Honest)
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))
rf.fit(X.iloc[train_idx], y.iloc[train_idx])
auc_grouped = roc_auc_score(y.iloc[test_idx], rf.predict_proba(X.iloc[test_idx])[:, 1])

print(f"Random Split ROC AUC (Dishonest): {auc_random:.3f}")
print(f"Grouped Split ROC AUC (Honest): {auc_grouped:.3f}")
print(f"Gap (Memorization penalty): {auc_random - auc_grouped:.3f}")


Random Split ROC AUC (Dishonest): 0.768
Grouped Split ROC AUC (Honest): 0.614
Gap (Memorization penalty): 0.154


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# Leakage Audit: Overlapping Windows
# The label 'is_declining_label' is derived from 'trend_direction', which compares the last 30 days to the previous 30 days.
# Using 'impressions_90d' and 'clicks_90d' means our features contain the label's window (the last 30 days).
# This is a classic "Future/overlapping windows" leak. Let's test the model without these leaky 90d aggregates.

leaky_features = [f for f in num_features if '90d' in f]
safe_num_features = [f for f in num_features if f not in leaky_features]

X_safe = df[safe_num_features + cat_features]

preprocessor_safe = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), safe_num_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), 
                          ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
    ])

rf_safe = Pipeline([
    ('preprocessor', preprocessor_safe),
    ('classifier', RandomForestClassifier(random_state=42, max_depth=10, n_estimators=100))
])

rf_safe.fit(X_safe.iloc[train_idx], y.iloc[train_idx])
auc_safe = roc_auc_score(y.iloc[test_idx], rf_safe.predict_proba(X_safe.iloc[test_idx])[:, 1])

print(f"Honest Split with LEAKY features ROC AUC: {auc_grouped:.3f}")
print(f"Honest Split with SAFE features ROC AUC: {auc_safe:.3f}")
print("\nConclusion: Removing the overlapping 90-day window features drops the score, confirming they were leaking future/current state information.")

# Let's look at a real failure example using the safe model
df_test = df.iloc[test_idx].copy()
df_test['safe_proba'] = rf_safe.predict_proba(X_safe.iloc[test_idx])[:, 1]
fp = df_test[(df_test['is_declining_label'] == 0)].sort_values('safe_proba', ascending=False).head(1)
print(f"\nHardest False Positive (Predicted decline, but stable/growing):")
print(f"Content {fp['content_id'].values[0]} | Proba: {fp['safe_proba'].values[0]:.3f} | Age: {fp['content_age_days'].values[0]}")


Honest Split with LEAKY features ROC AUC: 0.614
Honest Split with SAFE features ROC AUC: 0.588

Conclusion: Removing the overlapping 90-day window features drops the score, confirming they were leaking future/current state information.

Hardest False Positive (Predicted decline, but stable/growing):
Content content_ef6e7d7cfe15 | Proba: 0.864 | Age: 271


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# Old claim: "The Random Forest model accurately predicts which content will decline based on its 90-day impressions and position."
# Rewritten safe claim: "The Random Forest model was observed to directionally separate declining from non-declining content in this dataset. However, after removing features that overlapped with the label's time window, the predictive power decreased, suggesting that the model provides decision-support rather than guaranteed forecasts."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.